In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

1. Compute annualized geometric mean total returns for measures of US and European 10-year government bonds, public equity and high yield credit since 2000, and a measure of global commodities over the same period. Focus on cash instruments, where possible.

In [2]:
START = '2000-01-01'

# Fetch data via yfinance (adjusted close = total return for ETFs)
tickers = {
    '^TNX':    'US 10Y Yield',       # CBOE 10Y Treasury yield (for bond return calc)
    'SPY':     'US Equity',          # S&P 500 ETF (total return)
    'EZU':     'EU Equity',          # iShares MSCI EMU (total return)
    'HYG':     'US HY Credit',       # iShares iBoxx $ HY Corp Bond (total return)
    'EXHA.DE': 'EU HY Credit',       # iShares Euro HY Corp Bond (total return)
    'IBGL.L':  'EU 10Y Bond',        # iShares Euro Govt Bond 7-10yr (total return)
    'DBC':     'Commodities',        # Invesco DB Commodity Tracking (total return)
}

raw = yf.download(list(tickers.keys()), start=START, auto_adjust=True, progress=False)

# coverage
print("Data coverage:")
for ticker, name in tickers.items():
    col = raw['Close'][ticker].dropna()
    print(f"  {name:15s} ({ticker:8s}): {col.index[0].date()} to {col.index[-1].date()}  ({len(col)} daily obs)")

Data coverage:
  US 10Y Yield    (^TNX    ): 2000-01-03 to 2026-04-17  (6606 daily obs)
  US Equity       (SPY     ): 2000-01-03 to 2026-04-17  (6612 daily obs)
  EU Equity       (EZU     ): 2000-07-31 to 2026-04-17  (6467 daily obs)
  US HY Credit    (HYG     ): 2007-04-11 to 2026-04-17  (4786 daily obs)
  EU HY Credit    (EXHA.DE ): 2008-01-02 to 2026-04-17  (4644 daily obs)
  EU 10Y Bond     (IBGL.L  ): 2008-01-02 to 2026-04-17  (4619 daily obs)
  Commodities     (DBC     ): 2006-02-06 to 2026-04-17  (5081 daily obs)


In [3]:
# Compute monthly total returns

def bond_returns_from_yield(yield_series, maturity=10):
    """
    Approximate monthly total return of a constant-maturity bond from yield data.
    r_t ≈ y_{t-1}/12  -  D_mod(y_{t-1}) × Δy_t
    """
    y = yield_series.dropna() / 100  # percent → decimal
    y = y.resample('ME').last().dropna()
    y_prev = y.shift(1)
    dy = y - y_prev
    # Modified duration of a par bond with semiannual coupons
    n = maturity
    d_mac = ((1 + y_prev / 2) / y_prev) * (1 - 1 / (1 + y_prev / 2) ** (2 * n))
    d_mod = d_mac / (1 + y_prev / 2)
    ret = y_prev / 12 - d_mod * dy
    return ret.dropna()

def etf_monthly_returns(price_series):
    """Monthly total returns from adjusted-close ETF prices."""
    p = price_series.dropna().resample('ME').last().dropna()
    return p.pct_change().dropna()

# Build monthly return series for all 7 asset classes
close = raw['Close']

returns = pd.DataFrame({
    'US 10Y Bond':  bond_returns_from_yield(close['^TNX']),
    'EU 10Y Bond':  etf_monthly_returns(close['IBGL.L']),
    'US Equity':    etf_monthly_returns(close['SPY']),
    'EU Equity':    etf_monthly_returns(close['EZU']),
    'US HY Credit': etf_monthly_returns(close['HYG']),
    'EU HY Credit': etf_monthly_returns(close['EXHA.DE']),
    'Commodities':  etf_monthly_returns(close['DBC']),
})

print(f"Monthly returns: {returns.shape[0]} months × {returns.shape[1]} assets\n")
print("Per-series date range:")
for col in returns.columns:
    s = returns[col].dropna()
    print(f"  {col:15s}: {s.index[0]:%Y-%m} to {s.index[-1]:%Y-%m}  ({len(s)} months)")

Monthly returns: 315 months × 7 assets

Per-series date range:
  US 10Y Bond    : 2000-02 to 2026-04  (315 months)
  EU 10Y Bond    : 2008-02 to 2026-04  (219 months)
  US Equity      : 2000-02 to 2026-04  (315 months)
  EU Equity      : 2000-08 to 2026-04  (309 months)
  US HY Credit   : 2007-05 to 2026-04  (228 months)
  EU HY Credit   : 2008-02 to 2026-04  (219 months)
  Commodities    : 2006-03 to 2026-04  (242 months)


In [4]:
# Q1: Annualized geometric mean total returns since 2000

def ann_geo_mean(r):
    """Annualized geometric mean from monthly returns."""
    r = r.dropna()
    return np.exp(np.log(1 + r).mean() * 12) - 1

geo_means = returns.apply(ann_geo_mean)

q1_table = pd.DataFrame({
    'Annualized Geometric Mean Return (%)': (geo_means * 100).round(2),
})
q1_table.style.format('{:.2f}').set_caption('Q1: Annualized Geometric Mean Total Returns (since 2000)')

,Annualized Geometric Mean Return (%)
US 10Y Bond,3.62
EU 10Y Bond,0.86
US Equity,8.29
EU Equity,4.75
US HY Credit,4.92
EU HY Credit,0.46
Commodities,2.14


2. Compute the annualized volatility of all 7 returns. What things do you notice?

In [5]:
# Q2: Annualized volatility of all 7 return series

ann_vols = returns.std() * np.sqrt(12)

q2_table = pd.DataFrame({
    'Ann. Geo. Mean (%)':  (geo_means * 100).round(2),
    'Ann. Volatility (%)': (ann_vols * 100).round(2),
    'Return / Vol':        (geo_means / ann_vols).round(3),
})
q2_table.style.format({
    'Ann. Geo. Mean (%)': '{:.2f}',
    'Ann. Volatility (%)': '{:.2f}',
    'Return / Vol': '{:.3f}',
}).set_caption('Q2: Risk-Return Summary (since 2000)')

,Ann. Geo. Mean (%),Ann. Volatility (%),Return / Vol
US 10Y Bond,3.62,7.56,0.478
EU 10Y Bond,0.86,13.12,0.065
US Equity,8.29,15.16,0.547
EU Equity,4.75,21.06,0.225
US HY Credit,4.92,10.28,0.478
EU HY Credit,0.46,3.73,0.124
Commodities,2.14,18.54,0.115


**Observations:**

- US HY credit looks attractive on a return/vol basis with a return/vol ratio of ~0.48, matching US 10Y bonds. However, this is misleading because volatility understates the true risk of HY (fat tails, drawdown risk, equity-like crash exposure — addressed in Q7).
- US equity has the best risk-adjusted return (~0.55 return/vol), followed by US bonds and US HY credit, both at ~0.48.
- Bonds have substantially lower volatility than equities (~7–13% vs. ~15–21%), as expected. US 10Y bonds earn ~3.6% annualized with only ~7.6% vol.
- Commodities have high volatility (~18.5%) with modest geometric mean return (~2.1%), making them unattractive on a standalone basis. Their main value is as a diversifier.
- US assets outperform European assets across all three categories (bonds, equity, HY credit), both in absolute and risk-adjusted terms.
- The geometric mean penalizes volatile assets relative to the arithmetic mean — this volatility drag is largest for commodities and EU equity (both >18% vol).
- I noticed European asset series start later (2008) due to ETF launch dates, so their sample periods differ from US series (which start in 2000). This shorter window includes the GFC recovery and 2022 rate shock, which affects comparability.

3. Decompose the US equity and bond returns in (1) into carry (i.e. income) returns and price (i.e. capital) returns. You are not required to do this for HY credit. Why do you think I am not asking you to do this?

4. Compute the volatility of both income and capital returns for US equity and bonds. What things do you notice?

5. Assuming you generally invest in non-trivial portfolios (i.e. you have various investments in each investment portfolio), if you can be right at either asset allocation across asset classes or security selection within asset classes, but not both, which do you think is better to be correct in, and over what horizon? Justify your answer quantitatively.

6. What does that mean for a CIO deciding if he/she should be investing more resources in the asset allocation or security selection parts of his/her fund?

7. Based on your answer to questions (1) - (2), you should have determined that US HY credit appears, at least based on returns and volatility, like a desirable asset to hold. I would like you to assess if this is really true for a typical macro investor. Why or why not?
Justify your answer quantitatively. Things you should probably consider are: 

a. What is the right measure of risk for this asset? Is this risk priced by the market and if yes, how?

b. How is owning HY credit similar or different from dynamically managing a portfolio of government bonds and equity?

c. How time-varying are your answers to (a) and (b) above?

d. Are there practical investment constraints that may be relevant for some funds and could reduce their ability to invest in certain asset classes in ways that might be “optimal”? (e.g., leverage constraints, etc.)

e. How does being long HY credit compare with risk premium harvesting strategies more generally?

f. What are the implications of (d) and (e) for investor behavior as it pertains to HY credit?